In [66]:
import sys
sys.path.append('.')

from src.data_utils import load_data, prepare_dataset, split_data
from src.dataset import NextTokenDataset
import torch
from torch.utils.data import DataLoader

print("Все модули успешно импортированы!")

Все модули успешно импортированы!


In [67]:

df_raw = load_data('data/training.1600000.processed.noemoticon.csv', n_rows=100000)

df_clean = prepare_dataset(df_raw)

print("\nПримеры очищенных текстов:")
for i in range(3):
    print(f"  {i+1}. {df_clean['clean_text'].iloc[i]}")
    print(f"     Токены: {df_clean['tokens'].iloc[i]}")
    print()

Загрузка данных из data/training.1600000.processed.noemoticon.csv...
Загружено 100000 строк
Очистка текстов...
Токенизация...
Готово! 99828 примеров

Примеры очищенных текстов:
  1. a thats a bummer. you shoulda got david carr of third day to do it. d
     Токены: ['a', 'thats', 'a', 'bummer.', 'you', 'shoulda', 'got', 'david', 'carr', 'of', 'third', 'day', 'to', 'do', 'it.', 'd']

  2. is upset that he cant update his facebook by texting it... and might cry as a result school today also. blah!
     Токены: ['is', 'upset', 'that', 'he', 'cant', 'update', 'his', 'facebook', 'by', 'texting', 'it...', 'and', 'might', 'cry', 'as', 'a', 'result', 'school', 'today', 'also.', 'blah!']

  3. i dived many times for the ball. managed to save 50 the rest go out of bounds
     Токены: ['i', 'dived', 'many', 'times', 'for', 'the', 'ball.', 'managed', 'to', 'save', '50', 'the', 'rest', 'go', 'out', 'of', 'bounds']



In [59]:
train, val, test = split_data(df_clean)

print("\nПроверяем сохраненные файлы:")
!ls -la data/

Train: 79863 примеров
Val: 9982 примеров
Test: 9983 примеров

Проверяем сохраненные файлы:
total 515800
drwxr-xr-x@  7 tochi  staff        224 Mar 16 12:49 .
drwxr-xr-x@ 13 tochi  staff        416 Mar 15 14:13 ..
-rw-r--r--@  1 tochi  staff         50 Mar 15 14:38 file_from_visible_folders.txt
-rw-r--r--@  1 tochi  staff    2517052 Mar 16 12:49 test.csv
-rw-r--r--@  1 tochi  staff   20219003 Mar 16 12:49 train.csv
-rw-rw-r--@  1 tochi  staff  238803811 Mar 16 12:45 training.1600000.processed.noemoticon.csv
-rw-r--r--@  1 tochi  staff    2538328 Mar 16 12:49 val.csv


In [60]:
from torch.nn.utils.rnn import pad_sequence

def collate_batch(batch):
    inputs, targets = zip(*batch)

    inputs_padded = pad_sequence(inputs, batch_first=True, padding_value=0)
    targets_padded = pad_sequence(targets, batch_first=True, padding_value=0)
    
    return inputs_padded, targets_padded

train_dataset = NextTokenDataset('data/train.csv', max_len=20)
val_dataset = NextTokenDataset('data/val.csv', max_len=20)
test_dataset = NextTokenDataset('data/test.csv', max_len=20)

train_loader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True, 
    collate_fn=collate_batch,
    num_workers=0
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=32, 
    shuffle=False, 
    collate_fn=collate_batch,
    num_workers=0
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=32, 
    shuffle=False, 
    collate_fn=collate_batch,
    num_workers=0
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Размер словаря: 81295
Всего примеров после фильтрации: 77683
Размер словаря: 19986
Всего примеров после фильтрации: 9700
Размер словаря: 19547
Всего примеров после фильтрации: 9732

Train batches: 2428
Val batches: 304
Test batches: 305


In [ ]:
x, y = next(iter(train_loader))
print(f"Форма входных данных: {x.shape}") 
print(f"Форма целевых данных: {y.shape}") 
print(f"Размер словаря: {train_dataset.vocab_size}")
print(f"Индекс паддинга: {train_dataset.pad_idx}")

print(f"\nПервый пример входа: {x[0][:10]}...")
print(f"Первый пример цели: {y[0][:10]}...")

Форма входных данных: torch.Size([32, 19])
Форма целевых данных: torch.Size([32, 19])
Размер словаря: 81295
Индекс паддинга: 0

Первый пример входа: tensor([  792,  4117,  1959,    24,  3893,   423,   109,    16,   208, 13432])...
Первый пример цели: tensor([ 4117,  1959,    24,  3893,   423,   109,    16,   208, 13432,   292])...


In [62]:
from src.lstm_model import LSTMAutocomplete

vocab_size = train_dataset.vocab_size
pad_idx = train_dataset.pad_idx

model = LSTMAutocomplete(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_dim=256,
    num_layers=2,
    dropout=0.3,
    pad_idx=pad_idx
)

print(model)
print(f"\nВсего параметров: {sum(p.numel() for p in model.parameters())}")

LSTMAutocomplete(
  (embedding): Embedding(81295, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=81295, bias=True)
)

Всего параметров: 32220175


In [63]:
x, y = next(iter(train_loader))

with torch.no_grad():
    outputs = model(x)
    
print(f"Вход: {x.shape}")
print(f"Выход: {outputs.shape}")  # (batch_size, seq_len, vocab_size)
print(f"Логиты для первого токена первого примера: {outputs[0, 0, :5]}...")

Вход: torch.Size([32, 19])
Выход: torch.Size([32, 19, 81295])
Логиты для первого токена первого примера: tensor([ 0.0457,  0.0046,  0.0554, -0.0261, -0.0775])...


In [64]:
sample_idx = 0
sample_tokens = train_dataset.df.iloc[sample_idx]['token_list']
print(f"Полный текст: {' '.join(sample_tokens)}")

start_tokens = train_dataset.text_to_ids(sample_tokens[:3])
print(f"Начало (3 токена): {train_dataset.ids_to_text(start_tokens)}")

generated_ids = model.generate(start_tokens, max_new_tokens=5)
generated_text = train_dataset.ids_to_text(generated_ids)
print(f"Сгенерировано: {generated_text}")

Полный текст: another day at work working on the weekend sucks but the moneys pretty good. time to bite the bullet.
Начало (3 токена): another day at
Сгенерировано: another day at uggghhhh! reponding paaaaaaaaain kodis kodis


In [65]:
!zip -r data.zip data/ src/
print("Данные подготовлены!")

  adding: data/ (stored 0%)
  adding: data/val.csv (deflated 68%)
  adding: data/test.csv (deflated 68%)
  adding: data/training.1600000.processed.noemoticon.csv (deflated 64%)
  adding: data/train.csv (deflated 68%)
  adding: data/file_from_visible_folders.txt (stored 0%)
  adding: src/ (stored 0%)
  adding: src/dataset.py (deflated 64%)
  adding: src/data_utils.py (deflated 56%)
  adding: src/evaluate.py (deflated 70%)
  adding: src/lstm_model.py (deflated 71%)
  adding: src/train_lstm.py (deflated 64%)
Данные подготовлены!
